In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('VMT_2022v1_full_annual_20240226_monthly_18jun2024_nf_v4.csv', comment='#', low_memory=False)

col_headers = ['country_cd','region_cd','tribal_code','census_tract_cd','shape_id','scc','CD','MSR','activity_type',
               'ann_parm_value','calc_year','date_updated','data_set_id','jan_value','feb_value','mar_value','apr_value',
               'may_value','jun_value','jul_value','aug_value','sep_value','oct_value','nov_value','dec_value','comment']

df.columns = col_headers

month_cols = [
    'jan_value','feb_value','mar_value','apr_value','may_value','jun_value',
    'jul_value','aug_value','sep_value','oct_value','nov_value','dec_value'
]

In [3]:
df

,country_cd,region_cd,tribal_code,census_tract_cd,shape_id,scc,CD,MSR,activity_type,ann_parm_value,...,apr_value,may_value,jun_value,jul_value,aug_value,sep_value,oct_value,nov_value,dec_value,comment
0,US,1001,NaN,NaN,NaN,2201110300,NaN,NaN,VMT,3.139064e+06,...,3.256962e+05,3.887007e+05,2.786980e+05,2.793834e+05,2.696865e+05,2.836919e+05,3.266046e+05,1.838852e+05,1.837503e+05,NaN
1,US,1001,NaN,NaN,NaN,2201110400,NaN,NaN,VMT,1.447691e+05,...,1.502064e+04,1.792632e+04,1.285315e+04,1.288476e+04,1.243755e+04,1.308346e+04,1.506253e+04,8.480521e+03,8.474302e+03,NaN
2,US,1001,NaN,NaN,NaN,2201110500,NaN,NaN,VMT,3.178427e+06,...,3.297804e+05,3.935750e+05,2.821929e+05,2.828868e+05,2.730683e+05,2.872493e+05,3.307002e+05,1.861911e+05,1.860546e+05,NaN
3,US,1003,NaN,NaN,NaN,2201110200,NaN,NaN,VMT,3.681799e+06,...,3.820083e+05,4.559060e+05,3.268842e+05,3.276881e+05,3.163145e+05,3.327415e+05,3.830736e+05,2.156785e+05,2.155203e+05,NaN
4,US,1003,NaN,NaN,NaN,2201110300,NaN,NaN,VMT,1.201022e+07,...,1.246131e+06,1.487190e+06,1.066313e+06,1.068936e+06,1.031835e+06,1.085420e+06,1.249606e+06,7.035546e+05,7.030386e+05,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
258770,US,8123,NaN,NaN,NaN,2202620300,NaN,NaN,VMT,2.790690e+07,...,2.234974e+06,2.382738e+06,2.400617e+06,2.497147e+06,2.626448e+06,2.576440e+06,2.750407e+06,2.345195e+06,2.077753e+06,NaN
258771,US,8123,NaN,NaN,NaN,2202620400,NaN,NaN,VMT,1.597944e+07,...,1.279742e+06,1.364351e+06,1.374589e+06,1.429862e+06,1.503899e+06,1.475265e+06,1.574878e+06,1.342854e+06,1.189718e+06,NaN
258772,US,8123,NaN,NaN,NaN,2202620500,NaN,NaN,VMT,6.724186e+06,...,5.385186e+05,5.741223e+05,5.784304e+05,6.016894e+05,6.328444e+05,6.207951e+05,6.627126e+05,5.650764e+05,5.006361e+05,NaN
258773,US,8125,NaN,NaN,NaN,2202620300,NaN,NaN,VMT,4.144363e+06,...,3.319088e+05,3.538527e+05,3.565079e+05,3.708433e+05,3.900453e+05,3.826188e+05,4.084541e+05,3.482774e+05,3.085604e+05,NaN


In [4]:
reduction_pct = {
    "52": 0.2193,  # single-unit short haul
    "53": 0.1955,  # single-unit long haul
    "61": 0.1592,  # combo short haul
    "62": 0.2605,  # combo long haul
}

scale_factor = {k: (1.0 - v) for k, v in reduction_pct.items()}

In [5]:
# I only want to modify Cook County VMTs, so masking for FIPS == 17031 
df["src_type"] = df["scc"].astype(str).str.zfill(10).str[4:6]

mask_cook = df["region_cd"] == 17031
mask_type = df["src_type"].isin(scale_factor.keys())

mask = mask_cook & mask_type
print("Rows matched for scaling:", mask.sum())

Rows matched for scaling: 30


In [6]:
df.loc[mask, "ann_parm_value"] *= df.loc[mask, "src_type"].map(scale_factor)

for m in month_cols:
    df.loc[mask, m] *= df.loc[mask, "src_type"].map(scale_factor)

df = df.drop(columns=["src_type"])

In [10]:
vmt_cols = [
    'country_cd','region_cd','tribal_code','census_tract_cd','shape_id',
    'scc','CD','MSR','activity_type','ann_parm_value','calc_year',
    'date_updated','data_set_id','jan_value','feb_value','mar_value',
    'apr_value','may_value','jun_value','jul_value','aug_value',
    'sep_value','oct_value','nov_value','dec_value','comment'
]

# fields that MUST be quoted
ALWAYS_QUOTE = {
    "country_cd", "region_cd", "scc",
    "activity_type", "date_updated", "data_set_id"
}

def escape_csv_field(val: str) -> str:
    """
    Minimal CSV escaping for numeric or simple fields.
    Only quote when comma, quote, or newline is present.
    """
    s = "" if pd.isna(val) else str(val)
    if any(ch in s for ch in [',','"','\n','\r']):
        s = '"' + s.replace('"','""') + '"'
    return s

def force_quote(val: str) -> str:
    """
    Always wrap in quotes; escape inner quotes.
    """
    s = "" if pd.isna(val) else str(val)
    return '"' + s.replace('"','""') + '"'

def write_vmt_scaled(src_path, out_path, df_scaled, meta_lines):
    # ---- Load metadata prefix exactly as original ----
    with open(src_path, "r", encoding="utf-8") as f:
        meta = [next(f) for _ in range(meta_lines)]

    body = df_scaled.copy()

    # normalize key fields
    body["region_cd"] = body["region_cd"].astype(str).str.zfill(5)
    body["scc"]       = body["scc"].astype(str).str.zfill(10)

    # Force literal US
    body["country_cd"] = "US"

    with open(out_path, "w", encoding="utf-8") as f:
        # write metadata exactly as-is
        for line in meta:
            f.write(line)

        # Write body (NO HEADER)
        for _, row in body.iterrows():
            row_str = []
            for c in vmt_cols:
                val = row[c]

                if c in ALWAYS_QUOTE:
                    row_str.append(force_quote(val))
                else:
                    row_str.append(escape_csv_field(val))

            f.write(",".join(row_str) + "\n")

In [12]:
src_path  = "VMT_2022v1_full_annual_20240226_monthly_18jun2024_nf_v4.csv"
out_path  = "VMT_2022v1_full_annual_20240226_monthly_18jun2024_nf_v4_ZeroPre2010s.csv"
meta_lines = 14  # adjust if your file has more/less header comment lines

write_vmt_scaled(src_path, out_path, df, meta_lines)
